In [1]:
from datetime import datetime
import fastf1, pandas as pd, os

RAW_BASE  = "/tmp/f1_data/raw"

os.makedirs(f"{RAW_BASE}/dims", exist_ok=True)

drivers, teams = [], []
for year in range(2019, 2026):
    schedule = fastf1.get_event_schedule(year, include_testing=False)
    for _, event in schedule.iterrows():
        try:
            s = fastf1.get_session(year, event["RoundNumber"], "R")
            s.load(laps=False, telemetry=False, weather=False, messages=False)
            for _, r in s.results.iterrows():
                drivers.append({
                    "code": r["Abbreviation"], "full_name": r["FullName"]
                })
                teams.append({
                    "name": r["TeamName"]
                })
        except Exception as e:
            print(f"Pominięto {year} R1: {e}")

pd.DataFrame(drivers).drop_duplicates("code") \
  .to_parquet(f"{RAW_BASE}/dims/drivers_raw.parquet", index=False)
    
pd.DataFrame(teams).drop_duplicates("name") \
  .to_parquet(f"{RAW_BASE}/dims/teams_raw.parquet", index=False)
pd.DataFrame([
    {"name": "SOFT", "category": "DRY"},
    {"name": "MEDIUM", "category": "DRY"},
    {"name": "HARD", "category": "DRY"},
    {"name": "INTER", "category": "INTERMEDIATE"},
    {"name": "WET", "category": "WET"},
    {"name": "UNKNOWN", "category": "UNKNOWN"},
]).to_parquet(f"{RAW_BASE}/dims/compounds_raw.parquet", index=False)

print("Extracted: drivers, teams, compounds")

req         WARNING 	DEFAULT CACHE ENABLED! (6.02 MB) /home/michal/.cache/fastf1
core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['77', '44', '33', '5', '16', '20', '27', '7', '18', '26', '10', '4', '11', '23', '99', '63', '88', '8', '3', '55']
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['44', '77', '16', '33', '5', '4', '7', '10', '23', '11', '99', '26', '20', '18', '63', '88', '27', '3', '55', '8']
core           INFO 	Loading data for Chinese Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core          

Extracted: drivers, teams, compounds


In [2]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType
from pyspark.sql.functions import col

spark = SparkSession.builder \
        .appName("F1_Telemetry_Processing") \
        .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2") \
        .getOrCreate()

path = "/tmp/f1_data/raw/dims"
drivers_raw = spark.read.parquet(f"{path}/drivers_raw.parquet")
teams_raw = spark.read.parquet(f"{path}/teams_raw.parquet")
compounds_raw = spark.read.parquet(f"{path}/compounds_raw.parquet")

url = "jdbc:postgresql://localhost:5432/telemetry"
properties = {"user": "michal", "password": "root", "driver": "org.postgresql.Driver"}

drivers_raw.write.jdbc(url=url, table="dim_driver", mode="append", properties=properties)
teams_raw.write.jdbc(url=url, table="dim_team", mode="append", properties=properties)
compounds_raw.write.jdbc(url=url, table="dim_compound", mode="append", properties=properties)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/25 14:44:00 WARN Utils: Your hostname, michal-MS-7996, resolves to a loopback address: 127.0.1.1; using 192.168.0.58 instead (on interface enp2s0)
26/03/25 14:44:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/michal/.ivy2.5.2/cache
The jars for the packages stored in: /home/michal/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-95b31067-f963-4ba3-b3a7-631ee76c0d88;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.2 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 310ms :: artifacts dl 5ms
	:: modules in use:
	org.checkerfram